# Chapter 3 (leptonic) — Notebook 1: Neutrino reconstruction

**Goals**

- Solve the W-mass constraint $M_W^2 = (E_\ell + E_\nu)^2 - |\vec p_\ell + \vec p_\nu|^2$ for $p_z(\nu)$.
- Understand the two quadratic roots and the negative-discriminant case.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.1)
print('Number of events:', len(events))

## Background

MET gives only $p_x(\nu)$, $p_y(\nu)$. The longitudinal $p_z(\nu)$ follows from the W-mass 
constraint, a quadratic with up to two real roots. The helper [`topmass.neutrino`](../../topmass/neutrino.py) does the algebra.

In [ ]:
events = events[selection.semilep_preselection(events)]
lep = kinematics.leading_lepton(events)
met = kinematics.met_vector(events)
pz_plus, pz_minus, has_real = neutrino.solve_pz(lep.px, lep.py, lep.pz, lep.E, met.px, met.py)

print(f'Fraction with two real solutions: {has_real.mean():.2%}')
plt.hist(pz_plus[has_real],  bins=100, range=(-300, 300), alpha=0.5, label='pz+')
plt.hist(pz_minus[has_real], bins=100, range=(-300, 300), alpha=0.5, label='pz-')
plt.xlabel(r'$p_z(\nu)$ [GeV]'); plt.legend()

## ✏️ Your turn 1.1

▶️ Change the assumed W mass and re-run.

`has_real` is True when the W-mass quadratic has a real solution. This prints the fraction of events
with a **negative** discriminant (no real solution) for a chosen `M_W_TRY`. Physically, why can the
discriminant go negative (hint: the measured MET has finite resolution and can fluctuate)?

> **Stretch (optional):** run the cell for `M_W_TRY` = 78, then 83, and note how the fraction shifts.

In [ ]:
M_W_TRY = 80.4    # ✏️ try 78, 80.4, 83 GeV

pz_plus, pz_minus, has_real = neutrino.solve_pz(
    lep.px, lep.py, lep.pz, lep.E, met.px, met.py, m_w=M_W_TRY)
frac_neg = 1.0 - float(has_real.mean())
print(f'M_W = {M_W_TRY} GeV  ->  {frac_neg:.1%} of events have NO real solution (negative discriminant)')